In [2]:
# 구간 분할

import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv("./data/auto-mpg.csv", header=None)
df.columns = [
    "mpg",'cylinders','displacement',"horsepower","weight","acceleration","model year","origin","name"
]

In [4]:
df["horsepower"] = df["horsepower"].replace("?", np.nan)
df = df.dropna(subset=["horsepower"], axis=0)
df["horsepower"] = df["horsepower"].astype("float")

In [5]:
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,name
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino


In [6]:
df.info()

<class 'pandas.DataFrame'>
Index: 392 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           392 non-null    float64
 1   cylinders     392 non-null    int64  
 2   displacement  392 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        392 non-null    float64
 5   acceleration  392 non-null    float64
 6   model year    392 non-null    int64  
 7   origin        392 non-null    int64  
 8   name          392 non-null    str    
dtypes: float64(5), int64(3), str(1)
memory usage: 30.6 KB


In [7]:
count, bin_dividers = np.histogram(df["horsepower"],bins=3)
bin_dividers


array([ 46.        , 107.33333333, 168.66666667, 230.        ])

In [8]:
bin_names = ["저출력","보통출력","고출력"]

df["hp_bin"] = pd.cut(
    x=df["horsepower"],
    bins=bin_dividers,
    labels=bin_names,
    include_lowest=True
)

df[["horsepower","hp_bin"]].head(15)

,horsepower,hp_bin
0,130.0,보통출력
1,165.0,보통출력
2,150.0,보통출력
3,150.0,보통출력
4,140.0,보통출력
5,198.0,고출력
6,220.0,고출력
7,215.0,고출력
8,225.0,고출력
9,190.0,고출력


In [9]:
# 더미 변수
# pd.get_dummies()

horsepower_dummies = pd.get_dummies(df["hp_bin"])
horsepower_dummies.head(7)

,저출력,보통출력,고출력
0,False,True,False
1,False,True,False
2,False,True,False
3,False,True,False
4,False,True,False
5,False,False,True
6,False,False,True


In [10]:
df["hp_bin"].head(10)

0    보통출력
1    보통출력
2    보통출력
3    보통출력
4    보통출력
5     고출력
6     고출력
7     고출력
8     고출력
9     고출력
Name: hp_bin, dtype: category
Categories (3, str): ['저출력' < '보통출력' < '고출력']

In [11]:
horsepower_dummies_float = pd.get_dummies(df["hp_bin"],dtype=float)
horsepower_dummies_float

,저출력,보통출력,고출력
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,0.0,1.0,0.0
3,0.0,1.0,0.0
4,0.0,1.0,0.0
...,...,...,...
393,1.0,0.0,0.0
394,1.0,0.0,0.0
395,1.0,0.0,0.0
396,1.0,0.0,0.0


In [12]:
horsepower_dummies_drop = pd.get_dummies(df["hp_bin"],dtype=float,drop_first=True)
horsepower_dummies_drop

#보통출력도 저출력에 포함되어있어서

,보통출력,고출력
0,1.0,0.0
1,1.0,0.0
2,1.0,0.0
3,1.0,0.0
4,1.0,0.0
...,...,...
393,0.0,0.0
394,0.0,0.0
395,0.0,0.0
396,0.0,0.0


In [13]:
from sklearn import preprocessing

label_encoder = preprocessing.LabelEncoder()
onehot_encoder = preprocessing.OneHotEncoder()

onehot_labeled = label_encoder.fit_transform(df["hp_bin"].head(15))
print(onehot_labeled)
print(type(onehot_labeled))


[1 1 1 1 1 0 0 0 0 0 0 1 1 0 2]
<class 'numpy.ndarray'>


In [18]:
onehot_reshaped = onehot_labeled.reshape(len(onehot_labeled), 1)
print(onehot_reshaped)
print(type(onehot_reshaped))


[[1]
 [1]
 [1]
 [1]
 [1]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [1]
 [1]
 [0]
 [2]]
<class 'numpy.ndarray'>


In [21]:
onehot_fitted = onehot_encoder.fit_transform(onehot_reshaped)
print(onehot_fitted.toarray())
print(onehot_fitted)
print(type(onehot_fitted))



[[0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 15 stored elements and shape (15, 3)>
  Coords	Values
  (0, 1)	1.0
  (1, 1)	1.0
  (2, 1)	1.0
  (3, 1)	1.0
  (4, 1)	1.0
  (5, 0)	1.0
  (6, 0)	1.0
  (7, 0)	1.0
  (8, 0)	1.0
  (9, 0)	1.0
  (10, 0)	1.0
  (11, 1)	1.0
  (12, 1)	1.0
  (13, 0)	1.0
  (14, 2)	1.0
<class 'scipy.sparse._csr.csr_matrix'>
